In [1]:
###################################
# IMPORT
###################################
from polars import col, concat, lit, when
import polars as pl

import sys; sys.path.append('.'); 
import scripts.hdb_helpers as dc
import os



In [2]:
###################################
# DATA LOAD
###################################
hdbdata = pl.read_parquet('datalake/hdb/raw/hdbdata')
hdbdata.glimpse()
# Updated snapshot PROFILING done on date: #2026-07-17

Rows: 459007
Columns: 13
$ month               <str> '2000-01', '2000-01', '2000-01', '2000-01', '2000-01', '2000-01', '2000-01', '2000-01', '2000-01', '2000-01'
$ town                <str> 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO'
$ flat_type           <str> '3 ROOM', '3 ROOM', '3 ROOM', '3 ROOM', '3 ROOM', '3 ROOM', '3 ROOM', '3 ROOM', '3 ROOM', '3 ROOM'
$ block               <str> '170', '174', '216', '215', '218', '320', '320', '330', '330', '332'
$ street_name         <str> 'ANG MO KIO AVE 4', 'ANG MO KIO AVE 4', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 1'
$ storey_range        <str> '07 TO 09', '04 TO 06', '07 TO 09', '07 TO 09', '07 TO 09', '04 TO 06', '07 TO 09', '07 TO 09', '04 TO 06', '07 TO 09'
$ floor_area_sqm      <f64> 69.0, 61.0, 73.0, 73.0, 67.0, 73.0, 73.0, 68.

In [3]:
'''
REQUIREMENT:  Perform Data PROFILING on the dataset. You may code your own PROFILING rules or leverage on
open-source data quality frameworks.
'''

'''
We do first these three entry time PROFILING which might be better done self coded rather than packaged frameworoks and fuller summaries.
row counts -> column name and column types -> time variable
'''

'\nWe do first these three entry time PROFILING which might be better done self coded rather than packaged frameworoks and fuller summaries.\nrow counts -> column name and column types -> time variable\n'

In [4]:
####################################
# PROFILING: row count
####################################

# Q: How much row counts we are looking at.
row_count = (hdbdata
             .group_by('tabling_version')
             .len()
             .sort('tabling_version')
)
print(row_count)
# conclude: nothing alarming

shape: (3, 2)
┌─────────────────┬────────┐
│ tabling_version ┆ len    │
│ ---             ┆ ---    │
│ i32             ┆ u32    │
╞═════════════════╪════════╡
│ 2               ┆ 369651 │
│ 3               ┆ 52203  │
│ 4               ┆ 37153  │
└─────────────────┴────────┘


In [5]:
####################################
# PROFILING: column names and column types
####################################

# Q: What column names and what column types are we looking at?
columnDescribe = pl.DataFrame({
    'column': hdbdata.columns,
    'dtype': [str(dt) for dt in hdbdata.dtypes],
}).sort('dtype')

columnDescribe.show(100)
# conclude good: column names are all in snake case. 
# Consistent and good formatting helps ease of programming. no change required

# conclude: We only have 11 columns (excluding record_id and tabling_version) — 
# this is a narrow dataset, not a wide one with 20-50+ attributes)

# conclude: The only column name which potentially represents the time attribute for the dataset is [month].
# Note: if there were others, we will catch them in the profiling below.


# For the rest which do not represent the time attribute for the dataset -

# Q: are there any numerics which might be better as factors (String)
# ┌─────────────────────┬─────────┐
# │ column              ┆ dtype   │
# │ ---                 ┆ ---     │
# │ str                 ┆ str     │
# ╞═════════════════════╪═════════╡
# │ floor_area_sqm      ┆ Float64 │
# │ resale_price        ┆ Float64 │
# │ lease_commence_year ┆ Int64   │
# │ remaining_lease     ┆ Int64   │

# conclude: by context of column names, these are sensibly numeric, 
# assigned of the correct set of dtypes. No concerns.

# Q: are there any String type which should be numerical but is parsed as String?
# - regardless of the reason it was so.

# │ town                ┆ String  │
# │ flat_type           ┆ String  │
# │ block               ┆ String  │
# │ street_name         ┆ String  │
# │ storey_range        ┆ String  │
# │ flat_model          ┆ String  │

# conclude: by context of column names, these are sensibly textual, 
# assigned of the correct dtype of string. No concerns.

column,dtype
str,str
"""floor_area_sqm""","""Float64"""
"""resale_price""","""Float64"""
"""tabling_version""","""Int32"""
"""lease_commence_date""","""Int64"""
"""remaining_lease""","""Int64"""
"""month""","""String"""
"""town""","""String"""
"""flat_type""","""String"""
"""block""","""String"""


In [6]:
####################################
### PROFILING: time variable
####################################
# Let us profile the cleanliness of the time variable.
# then let us profile the row count per month

monthvalues = (hdbdata
             .group_by('month', maintain_order=True)
             .len()
)

In [7]:
# Q: What is the format of the time variable
dc.showall(monthvalues)
# conclude: it specifies the month of the record.
# conclude good: the year-month format is in one consistent form - no variances.
# conclude good: The format is immediately sortable into earlier and later, unlike formats like January 2013
# Downstream validation: Let month be only allowed input format 'YYYY-MM'.

shape: (204, 2)
┌─────────┬──────┐
│ month   ┆ len  │
│ ---     ┆ ---  │
│ str     ┆ u32  │
╞═════════╪══════╡
│ 2000-01 ┆ 2386 │
│ 2000-02 ┆ 2313 │
│ 2000-03 ┆ 2467 │
│ 2000-04 ┆ 2548 │
│ 2000-05 ┆ 3230 │
│ 2000-06 ┆ 3283 │
│ 2000-07 ┆ 2988 │
│ 2000-08 ┆ 3166 │
│ 2000-09 ┆ 3088 │
│ 2000-10 ┆ 3231 │
│ 2000-11 ┆ 3324 │
│ 2000-12 ┆ 2838 │
│ 2001-01 ┆ 2790 │
│ 2001-02 ┆ 2885 │
│ 2001-03 ┆ 3147 │
│ 2001-04 ┆ 3092 │
│ 2001-05 ┆ 3106 │
│ 2001-06 ┆ 3329 │
│ 2001-07 ┆ 3398 │
│ 2001-08 ┆ 3244 │
│ 2001-09 ┆ 3220 │
│ 2001-10 ┆ 3490 │
│ 2001-11 ┆ 3349 │
│ 2001-12 ┆ 3005 │
│ 2002-01 ┆ 3385 │
│ 2002-02 ┆ 2633 │
│ 2002-03 ┆ 3115 │
│ 2002-04 ┆ 3300 │
│ 2002-05 ┆ 3298 │
│ 2002-06 ┆ 2836 │
│ 2002-07 ┆ 3282 │
│ 2002-08 ┆ 2829 │
│ 2002-09 ┆ 2749 │
│ 2002-10 ┆ 3156 │
│ 2002-11 ┆ 2831 │
│ 2002-12 ┆ 2684 │
│ 2003-01 ┆ 2705 │
│ 2003-02 ┆ 1858 │
│ 2003-03 ┆ 2270 │
│ 2003-04 ┆ 2160 │
│ 2003-05 ┆ 2272 │
│ 2003-06 ┆ 2162 │
│ 2003-07 ┆ 2273 │
│ 2003-08 ┆ 2195 │
│ 2003-09 ┆ 2423 │
│ 2003-10 ┆ 2849 │
│ 2003-11 ┆ 280

In [8]:
# Q: Is there any anomalous row counts in any month?
monthvalues['len'].describe()
# conclude: Average records in a month = 2273
# Max records in a month = 3679, nothing alarmig yet
# Min records in a month: 886, nothing alarming yet
# Note: min was coming not from earliest the month in the records. 
# Hence it is not due to cutoff in early data collection. nothing alarming.
# Nothing alarming overall.

statistic,value
str,f64
"""count""",204.0
"""null_count""",0.0
"""mean""",2250.034314
"""std""",624.331993
"""min""",886.0
"""25%""",1747.0
"""50%""",2273.0
"""75%""",2684.0
"""max""",3679.0


In [9]:
# Q: what is the time span of the dataset:
min_max_month = (hdbdata
               .sort('tabling_version')
               .group_by('tabling_version')
               .agg(
                 pl.min('month').alias('1st_month'),
                 pl.max('month').alias('last_month'),
             )
)

print(min_max_month)
# conclude good: tables 1,2,3 are linked in continuity. No concerns.
# conclude: earliest data is 2000-01 latest data is 2016-12.
# downstream action: will scope it out to only 2012-01 onwards for table 2. In script 2_data_cleaning.

shape: (3, 3)
┌─────────────────┬───────────┬────────────┐
│ tabling_version ┆ 1st_month ┆ last_month │
│ ---             ┆ ---       ┆ ---        │
│ i32             ┆ str       ┆ str        │
╞═════════════════╪═══════════╪════════════╡
│ 2               ┆ 2000-01   ┆ 2012-02    │
│ 3               ┆ 2012-03   ┆ 2014-12    │
│ 4               ┆ 2015-01   ┆ 2016-12    │
└─────────────────┴───────────┴────────────┘


In [10]:
# Q: Are there any gaps in the monthly time series?
data_exists_monthdate = (
    hdbdata
    .select('month').unique()
    .with_columns(monthdate = col('month').str.strptime(pl.Date, '%Y-%m'))
    .sort('monthdate')
    .with_columns(in_data = lit(1))
)


complete_monthdate_list = (
    pl.date_range(
        data_exists_monthdate['monthdate'].min(),
        data_exists_monthdate['monthdate'].max(),
        interval='1mo',
        eager=True,
    )
    .alias('monthdate').to_frame()
)

monthdate_indicator = complete_monthdate_list.join(data_exists_monthdate, on='monthdate', how='left')

In [11]:
dc.showall(monthdate_indicator)
# conclude good: By full visual check, confirm every month is in the datas. Also every month format is correct.

shape: (204, 3)
┌────────────┬─────────┬─────────┐
│ monthdate  ┆ month   ┆ in_data │
│ ---        ┆ ---     ┆ ---     │
│ date       ┆ str     ┆ i32     │
╞════════════╪═════════╪═════════╡
│ 2000-01-01 ┆ 2000-01 ┆ 1       │
│ 2000-02-01 ┆ 2000-02 ┆ 1       │
│ 2000-03-01 ┆ 2000-03 ┆ 1       │
│ 2000-04-01 ┆ 2000-04 ┆ 1       │
│ 2000-05-01 ┆ 2000-05 ┆ 1       │
│ 2000-06-01 ┆ 2000-06 ┆ 1       │
│ 2000-07-01 ┆ 2000-07 ┆ 1       │
│ 2000-08-01 ┆ 2000-08 ┆ 1       │
│ 2000-09-01 ┆ 2000-09 ┆ 1       │
│ 2000-10-01 ┆ 2000-10 ┆ 1       │
│ 2000-11-01 ┆ 2000-11 ┆ 1       │
│ 2000-12-01 ┆ 2000-12 ┆ 1       │
│ 2001-01-01 ┆ 2001-01 ┆ 1       │
│ 2001-02-01 ┆ 2001-02 ┆ 1       │
│ 2001-03-01 ┆ 2001-03 ┆ 1       │
│ 2001-04-01 ┆ 2001-04 ┆ 1       │
│ 2001-05-01 ┆ 2001-05 ┆ 1       │
│ 2001-06-01 ┆ 2001-06 ┆ 1       │
│ 2001-07-01 ┆ 2001-07 ┆ 1       │
│ 2001-08-01 ┆ 2001-08 ┆ 1       │
│ 2001-09-01 ┆ 2001-09 ┆ 1       │
│ 2001-10-01 ┆ 2001-10 ┆ 1       │
│ 2001-11-01 ┆ 2001-11 ┆ 1       │
│ 20

In [12]:
check_12_months_within_year = (
    monthdate_indicator
    .with_columns(year=col('monthdate').dt.year())
    .group_by('year', maintain_order=True)
    .len()
)
dc.showall(check_12_months_within_year)
# conclude good: By more concise check, all years has all and only 12 months, => every month is in the data.

# conclude good: no gaps reported.

print('row counts -> column name and column types -> time variable')
print('In these three specifics, no concerns were yet raised, so there is nothing to triage.')

shape: (17, 2)
┌──────┬─────┐
│ year ┆ len │
│ ---  ┆ --- │
│ i32  ┆ u32 │
╞══════╪═════╡
│ 2000 ┆ 12  │
│ 2001 ┆ 12  │
│ 2002 ┆ 12  │
│ 2003 ┆ 12  │
│ 2004 ┆ 12  │
│ 2005 ┆ 12  │
│ 2006 ┆ 12  │
│ 2007 ┆ 12  │
│ 2008 ┆ 12  │
│ 2009 ┆ 12  │
│ 2010 ┆ 12  │
│ 2011 ┆ 12  │
│ 2012 ┆ 12  │
│ 2013 ┆ 12  │
│ 2014 ┆ 12  │
│ 2015 ┆ 12  │
│ 2016 ┆ 12  │
└──────┴─────┘
row counts -> column name and column types -> time variable
In these three specifics, no concerns were yet raised, so there is nothing to triage.


In [13]:
'''
We now deal with categorical and numerical data profiling
We will now use an external tool which provides more details for lesser work.
We will use the ProfileReport function from the ydata-profiling package to easier and quickly generate the views we need.
'''

'''
# Author note. I  understand the project requests to do work programmatically as much as possible;
# However, i wish to use also ydata_profile to showcase that i am capable in 
# a) leverage [open-source data quality frameworks] and  
# b) integrating out-of-code processes with pipelines as seamlessly as I can.
'''

'\n# Author note. I  understand the project requests to do work programmatically as much as possible;\n# However, i wish to use also ydata_profile to showcase that i am capable in \n# a) leverage [open-source data quality frameworks] and  \n# b) integrating out-of-code processes with pipelines as seamlessly as I can.\n'

In [14]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    hdbdata.to_pandas(),
    title='HDB Resale Master Dataset - Data Profile',
)

os.makedirs('output', exist_ok=True)
profile.to_file('output/hdb_ydataprofile_report.html')

/Users/murftech/Root/production/hdb_assess_submission/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/var/folders/b2/jxsmc4xx49v5lrwhc889dxt00000gn/T/ipykernel_55396/493005442.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|                                                                                                                                      | 0/5 [00:00<?, ?it/s]

Summarize dataset:   0%|                                                                                                           | 0/18 [00:00<?, ?it/s, Describe variable: month]

Summarize dataset:   0%|                                                                                                     | 0/18 [00:00<?, ?it/s, Describe variable: street_name]

...

Render HTML:   0%|                                                                                                                                            | 0/1 [00:00<?, ?it/s]

Render HTML: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.52it/s]

Render HTML: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.48it/s]

Export report to file:   0%|                                                                                                                                  | 0/1 [00:00<?, ?it/s]

Export report to file: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 152.65it/s]

In [15]:
# openfile = 'false'
openfile = 'true'
if openfile == 'true':
    import os
    os.system('open output/hdb_ydataprofile_report.html')

In [16]:
'''
# Becasue the report of this tool is static file based, 
# we need to take screenshots portions from the report and give the profiling decision in-line 
# in a file type like microsoft word docx.
# i have done the exercise in this document: output/hdb_ydataprofile_report_downstream.docx
# To review the decisions, please open the docx file if you already have an associated app installed - with the command below:
'''

# openfile = 'false'
openfile = 'true'
if openfile == 'true':
    import os
    os.system('open output/hdb_ydataprofile_report_downstream.docx')

In [17]:
###################################
# Profiling: Categorical Variable
###################################

# Categorical variable: 
# Cardinality Q1: does the number of distinct values make sense. Is the only all-applicable question.
# Cardinality Q2: Does any of them became near to a row key?
# String hiding the type? Which should be a number, numeric, time but got saved as a string in source.
# Long Tails, of potentially inconsequential values.
# Format/Casing Inconsistency
# Is it a valid member 

In [18]:
'''
Followup from: output/hdb_ydataprofile_report_downstream.docx

# storey_range
# Downstream profiling: We will investigate in code if all values are of this pattern.

# flat_model
# Downstream profiling: We will investigate in code if there are long tail of rare values, 
# which we then ask if they are candidates for merging with other 
'''

'\nFollowup from: output/hdb_ydataprofile_report_downstream.docx\n\n# storey_range\n# Downstream profiling: We will investigate in code if all values are of this pattern.\n\n# flat_model\n# Downstream profiling: We will investigate in code if there are long tail of rare values, \n# which we then ask if they are candidates for merging with other \n'

In [19]:
# storey_range
# Downstream profiling: We will investigate in code if all values are of this pattern dd TO dd.
hdbdata['storey_range'].value_counts().sort('count').show(1000)
# conclude: Yes. 
# Downstream validation: Let storey_range be only allowed input format as such eg '28 TO 30'.

storey_range,count
str,u32
"""31 TO 35""",2
"""49 TO 51""",2
"""36 TO 40""",7
"""43 TO 45""",8
"""46 TO 48""",8
"""26 TO 30""",39
"""40 TO 42""",50
"""21 TO 25""",92
"""37 TO 39""",103


In [20]:
# flat_model
# Downstream profiling: We will investigate in code if there are long tail of rare values, 
# which we then ask if they are candidates for merging with others

hdbdata['flat_model'].value_counts().sort('count').show(1000)
# ANS: yes there are.

# ANS: Several of these are really variants of the same base design rather than fully independent categories, 
# So that they're natural merge candidates if we look for another grouping with fewer buckets:
# However this is a decision to be made later by stakeholders. 
# Since they are all real HDB flat_model names, there is no need to edit this column.

# ┌────────────────────────┬────────┐
# │ flat_model             ┆ count  │
# │ ---                    ┆ ---    │
# │ str                    ┆ u32    │
# ╞════════════════════════╪════════╡
# │ Premium Apartment Loft ┆ 5      │ Potentially a rare value
# │ 2-room                 ┆ 17     │ Potentially a rare value
# │ Type S2                ┆ 55     │ Potentially a rare value
# │ ImPROVEd-Maisonette    ┆ 56     │ Potentially a rare value
# │ Premium Maisonette     ┆ 72     │ Potentially a rare value
# │ Type S1                ┆ 138    │ 
# │ Multi Generation       ┆ 186    │
# │ DBSS                   ┆ 277    │
# │ Terrace                ┆ 349    │
# │ Model A-Maisonette     ┆ 771    │
# │ Adjoined flat          ┆ 933    │
# │ Model A2               ┆ 8064   │
# │ Maisonette             ┆ 12318  │
# │ Apartment              ┆ 18846  │
# │ Standard               ┆ 20219  │
# │ Premium Apartment      ┆ 26340  │
# │ Simplified             ┆ 27334  │
# │ New Generation         ┆ 87611  │
# │ ImPROVEd               ┆ 123696 │
# │ Model A                ┆ 131720 │
# └────────────────────────┴────────┘

flat_model,count
str,u32
"""Premium Apartment Loft""",5
"""2-room""",17
"""Type S2""",55
"""Improved-Maisonette""",56
"""Premium Maisonette""",72
"""Type S1""",138
"""Multi Generation""",186
"""DBSS""",277
"""Terrace""",349


In [21]:


###################################
# Profiling: Categorical Variable
###################################

# Numeric variable: 



hdbdata

month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,tabling_version,remaining_lease,record_id
str,str,str,str,str,str,f64,str,i64,f64,i32,i64,str
"""2000-01""","""ANG MO KIO""","""3 ROOM""","""170""","""ANG MO KIO AVE 4""","""07 TO 09""",69.0,"""Improved""",1986,147000.0,2,null,"""002-000000"""
"""2000-01""","""ANG MO KIO""","""3 ROOM""","""174""","""ANG MO KIO AVE 4""","""04 TO 06""",61.0,"""Improved""",1986,144000.0,2,null,"""002-000001"""
"""2000-01""","""ANG MO KIO""","""3 ROOM""","""216""","""ANG MO KIO AVE 1""","""07 TO 09""",73.0,"""New Generation""",1976,159000.0,2,null,"""002-000002"""
"""2000-01""","""ANG MO KIO""","""3 ROOM""","""215""","""ANG MO KIO AVE 1""","""07 TO 09""",73.0,"""New Generation""",1976,167000.0,2,null,"""002-000003"""
"""2000-01""","""ANG MO KIO""","""3 ROOM""","""218""","""ANG MO KIO AVE 1""","""07 TO 09""",67.0,"""New Generation""",1976,163000.0,2,null,"""002-000004"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""2016-12""","""YISHUN""","""5 ROOM""","""297""","""YISHUN ST 20""","""13 TO 15""",112.0,"""Improved""",2000,488000.0,4,82,"""004-459002"""
"""2016-12""","""YISHUN""","""5 ROOM""","""838""","""YISHUN ST 81""","""01 TO 03""",122.0,"""Improved""",1987,455000.0,4,69,"""004-459003"""
"""2016-12""","""YISHUN""","""EXECUTIVE""","""664""","""YISHUN AVE 4""","""10 TO 12""",181.0,"""Apartment""",1992,778000.0,4,74,"""004-459004"""


In [22]:
###################################
# Profiling: Numeric Variable - resale_price IQR by year and flat_type
###################################

'''
RQUIREMENT: Identify any potentially anomalous resale price based on any heuristics you deem
appropriate. Please document your heuristic and assumptions in your documentations.
'''

## There is a most robust quick method to have a first flag. That is the 1.5*IQR fence method.
## We assume that prices could increase or drop by year. Year to be taken as consideration.
## We assume that prices of bigger flats are more expensive than smaller flats. flat_type to be taken into consideration.
## within each cohort of year and flattype, we apply 1.5*IQR to quickly flag out potential anomalous resale prices.

'\nRQUIREMENT: Identify any potentially anomalous resale price based on any heuristics you deem\nappropriate. Please document your heuristic and assumptions in your documentations.\n'

In [23]:
hdb_sel = hdbdata.select('month', 'record_id', 'flat_type', 'resale_price')

resale_price_iqr = (
    hdb_sel
    .with_columns(year=col('month').str.slice(0, 4))
    .with_columns(
        group_samplesize = pl.len().over(['year', 'flat_type']),
        q1=col('resale_price').quantile(0.25, interpolation='linear').over(['year', 'flat_type']),
        group_median = pl.median('resale_price').over(['year', 'flat_type']),
        q3=col('resale_price').quantile(0.75, interpolation='linear').over(['year', 'flat_type']),
    )
    .with_columns(iqr=col('q3') - col('q1'))
    .with_columns(
        upp_1_5_iqr=col('q1') - 1.5 * col('iqr'),
        low_1_5_iqr=col('q3') + 1.5 * col('iqr'),
    )
    .with_columns(
        is_outlier=(col('resale_price') < col('upp_1_5_iqr')) | (col('resale_price') > col('low_1_5_iqr'))
    )
)

dc.sortcount(resale_price_iqr, 'is_outlier')
# validate
dc.showall(resale_price_iqr.filter(col('is_outlier')==True), 'closey')

459007
shape: (2, 3)
┌────────────┬────────┬──────────┐
│ is_outlier ┆ count  ┆ pcnt     │
│ ---        ┆ ---    ┆ ---      │
│ bool       ┆ u32    ┆ f64      │
╞════════════╪════════╪══════════╡
│ false      ┆ 438771 ┆ 0.955914 │
│ true       ┆ 20236  ┆ 0.044086 │
└────────────┴────────┴──────────┘
shape: (20_236, 13)
┌─────────┬────────────┬───────────┬──────────────┬──────┬──────────────────┬──────────┬──────────────┬──────────┬──────────┬─────────────┬─────────────┬────────────┐
│ month   ┆ record_id  ┆ flat_type ┆ resale_price ┆ year ┆ group_samplesize ┆ q1       ┆ group_median ┆ q3       ┆ iqr      ┆ upp_1_5_iqr ┆ low_1_5_iqr ┆ is_outlier │
│ ---     ┆ ---        ┆ ---       ┆ ---          ┆ ---  ┆ ---              ┆ ---      ┆ ---          ┆ ---      ┆ ---      ┆ ---         ┆ ---         ┆ ---        │
│ str     ┆ str        ┆ str       ┆ f64          ┆ str  ┆ u32              ┆ f64      ┆ f64          ┆ f64      ┆ f64      ┆ f64         ┆ f64         ┆ bool       │
╞═════════╪

In [24]:
# Q: How many records fall outside their cohort's 1.5*IQR fence, overall and by group, if we use simple IQR fence?
outlier_summary = (
    resale_price_iqr
    .group_by('year', maintain_order=True)
    .agg(
        n=pl.len(),
        n_outlier=col('is_outlier').sum(),
    )
    .with_columns(pcnt_outlier=(col('n_outlier') / col('n')))
    .sort(['year'])
)

dc.showall(outlier_summary)


print(f"total records flagged as outlier: {resale_price_iqr['is_outlier'].sum():,} "
      f"out of {resale_price_iqr.height:,} ({resale_price_iqr['is_outlier'].mean():.2%})")

# conclude:
# downstream action: decide stakeholders how could we further the rules to decide on flagging outlier or even dropping data.
# in script 2_data_validation
# no action taken as of now

shape: (17, 4)
┌──────┬───────┬───────────┬──────────────┐
│ year ┆ n     ┆ n_outlier ┆ pcnt_outlier │
│ ---  ┆ ---   ┆ ---       ┆ ---          │
│ str  ┆ u32   ┆ u32       ┆ f64          │
╞══════╪═══════╪═══════════╪══════════════╡
│ 2000 ┆ 34862 ┆ 994       ┆ 0.028512     │
│ 2001 ┆ 38055 ┆ 993       ┆ 0.026094     │
│ 2002 ┆ 36098 ┆ 1000      ┆ 0.027702     │
│ 2003 ┆ 29003 ┆ 775       ┆ 0.026721     │
│ 2004 ┆ 29112 ┆ 753       ┆ 0.025866     │
│ 2005 ┆ 30045 ┆ 1019      ┆ 0.033916     │
│ 2006 ┆ 27427 ┆ 1440      ┆ 0.052503     │
│ 2007 ┆ 26982 ┆ 1609      ┆ 0.059632     │
│ 2008 ┆ 27262 ┆ 1310      ┆ 0.048052     │
│ 2009 ┆ 30482 ┆ 1656      ┆ 0.054327     │
│ 2010 ┆ 34854 ┆ 1629      ┆ 0.046738     │
│ 2011 ┆ 22281 ┆ 1136      ┆ 0.050985     │
│ 2012 ┆ 23198 ┆ 1343      ┆ 0.057893     │
│ 2013 ┆ 16097 ┆ 1024      ┆ 0.063614     │
│ 2014 ┆ 16096 ┆ 934       ┆ 0.058027     │
│ 2015 ┆ 17780 ┆ 1224      ┆ 0.068841     │
│ 2016 ┆ 19373 ┆ 1397      ┆ 0.072111     │
└──────┴───────┴─

In [25]:
'''
END OF SCRIPT.
We have gathered all downstream work, tagged and ctrl+f by downstream.
There is no DATA WRITE in this script. It is also not required in the operationalized pipeline.
It should however be ran out of AWS, on a regular basis, including writing further profiling codes to check for data drifts, new validations required.
'''

'\nEND OF SCRIPT.\nWe have gathered all downstream work, tagged and ctrl+f by downstream.\nThere is no DATA WRITE in this script. It is also not required in the operationalized pipeline.\nIt should however be ran out of AWS, on a regular basis, including writing further profiling codes to check for data drifts, new validations required.\n'